## this is checking s3 by comapring the files available before aquaduct 4.0 merge

In [39]:
import boto3
import pandas as pd
from io import BytesIO
from botocore.exceptions import ClientError

# Initialize S3 client using your configured AWS credentials
s3 = boto3.client("s3")

# Update these to match your team's S3 location
BUCKET = "atlas-acm-research-f26"
PREFIX = "dataset/"  # Use "" if files are at bucket root

files = [
    "im3_baseline.csv",
    "im3_eia_combined.csv",
    "im3_multimodal_atlas_fueltype.csv",
    "im3_multimodal_atlas_retail.csv"
]

datasets = {}

for filename in files:
    key = f"{PREFIX}{filename}"

    try:
        response = s3.get_object(
            Bucket=BUCKET,
            Key=key
        )

        df = pd.read_csv(
            BytesIO(response["Body"].read()),
            dtype={"id": str}
        )

        datasets[filename] = df

        print(f"\n{'=' * 65}")
        print(f"FILE: {filename}")
        print(f"{'=' * 65}")

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        if "id" in df.columns:
            print("Unique IDs:", df["id"].nunique())
            print("Duplicate IDs:", df["id"].duplicated().sum())
            print("Missing IDs:", df["id"].isna().sum())
        else:
            print("WARNING: No 'id' column found")

        print("\nFirst 3 rows:")
        display(df.head(3))

    except ClientError as e:
        print(f"Error accessing {key}: {e}")


FILE: im3_baseline.csv
Shape: (1147, 13)
Columns: ['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type']
Unique IDs: 1147
Duplicate IDs: 0
Missing IDs: 0

First 3 rows:


,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,lon,lat,type
0,0002744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,-74.496521,40.544256,building
1,0007805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,-82.814358,40.100657,building
2,0009474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,-81.546515,35.894738,campus



FILE: im3_eia_combined.csv
Shape: (1147, 50)
Columns: ['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type', 'nearest_plant_km', 'plants_50km', 'capacity_50km_mw', 'renewable_50km_mw', 'renewable_share_50km', 'nearest_big_plant_km', 'ba_code', 'ba_peak_mw', 'ba_avg_mw', 'ba_peak_to_avg', 'ba_demand_coverage', 'ba_share_coal', 'ba_share_gas', 'ba_share_nuclear', 'ba_share_hydro', 'ba_share_solar', 'ba_share_wind', 'ba_share_fossil', 'ba_share_renewable', 'ba_share_clean', 'ba_total_gen_mwh', 'comm_elec_price_cents_kwh', 'comm_elec_sales_gwh', 'st_share_coal', 'st_share_gas', 'st_share_nuclear', 'st_share_hydro', 'st_share_solar', 'st_share_wind', 'st_share_fossil', 'st_share_renewable', 'st_share_clean', 'st_total_gen_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd', 'size_class', 'period']
Unique IDs: 1147
Duplicate IDs: 0
Missing IDs: 0

First 3 rows:


,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,...,st_share_solar,st_share_wind,st_share_fossil,st_share_renewable,st_share_clean,st_total_gen_mwh,est_mwh_per_hour,est_hourly_energy_cost_usd,size_class,period
0,0002744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,...,0.032733,0.000334,0.491908,0.033171,0.491695,6.088605e+07,10.5786,2027.91762,2,2026-07
1,0007805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,...,0.053625,0.020179,0.805069,0.076741,0.193017,1.474231e+08,18.8209,2646.21854,2,2026-07
2,0009474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,...,0.094154,0.006913,0.533911,0.135397,0.453233,1.354369e+08,340.7194,36559.19162,2,2026-07



FILE: im3_multimodal_atlas_fueltype.csv
Shape: (1147, 21)
Columns: ['id', 'state_abb', 'county', 'type', 'sqft', 'lat', 'lon', 'comm_elec_price_cents_kwh', 'comm_elec_sales_mwh', 'st_share_coal', 'st_share_gas', 'st_share_nuclear', 'st_share_hydro', 'st_share_solar', 'st_share_wind', 'st_share_fossil', 'st_share_renewable', 'st_share_clean', 'st_total_gen_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd']
Unique IDs: 1147
Duplicate IDs: 0
Missing IDs: 0

First 3 rows:


,id,state_abb,county,type,sqft,lat,lon,comm_elec_price_cents_kwh,comm_elec_sales_mwh,st_share_coal,...,st_share_nuclear,st_share_hydro,st_share_solar,st_share_wind,st_share_fossil,st_share_renewable,st_share_clean,st_total_gen_mwh,est_mwh_per_hour,est_hourly_energy_cost_usd
0,0002744301,NJ,Middlesex County,building,105786.0,40.544256,-74.496521,19.17,3445.43308,0.000000,...,0.458524,0.000104,0.032733,0.000334,0.491908,0.033171,0.491695,6.088605e+07,10.5786,2027.91762
1,0007805491,OH,Franklin County,building,188209.0,40.100657,-82.814358,14.06,6324.99033,0.243225,...,0.116277,0.002936,0.053625,0.020179,0.805069,0.076741,0.193017,1.474231e+08,18.8209,2646.21854
2,0009474864,NC,Caldwell County,campus,3407194.0,35.894738,-81.546515,10.73,5092.11716,0.132180,...,0.317836,0.034330,0.094154,0.006913,0.533911,0.135397,0.453233,1.354369e+08,340.7194,36559.19162



FILE: im3_multimodal_atlas_retail.csv
Shape: (1147, 12)
Columns: ['id', 'sqft', 'lat', 'lon', 'type', 'size_class', 'state_abb', 'period', 'comm_elec_price_cents_kwh', 'comm_elec_sales_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd']
Unique IDs: 1147
Duplicate IDs: 0
Missing IDs: 0

First 3 rows:


,id,sqft,lat,lon,type,size_class,state_abb,period,comm_elec_price_cents_kwh,comm_elec_sales_mwh,est_mwh_per_hour,est_hourly_energy_cost_usd
0,2744301,105786.0,40.544256,-74.496521,building,2,NJ,2026-07,19.17,3445.43308,10.5786,2027.91762
1,7805491,188209.0,40.100657,-82.814358,building,2,OH,2026-07,14.06,6324.99033,18.8209,2646.21854
2,9474864,3407194.0,35.894738,-81.546515,campus,2,NC,2026-07,10.73,5092.11716,340.7194,36559.19162


## comparing the datasets

In [40]:
summary = []

for filename, df in datasets.items():
    summary.append({
        "dataset": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "unique_ids": df["id"].nunique() if "id" in df else None,
        "duplicate_ids": df["id"].duplicated().sum() if "id" in df else None,
        "has_lat": any("lat" in c.lower() for c in df.columns),
        "has_lon": any(
            "lon" in c.lower() or "lng" in c.lower()
            for c in df.columns
        )
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,dataset,rows,columns,unique_ids,duplicate_ids,has_lat,has_lon
0,im3_baseline.csv,1147,13,1147,0,True,True
1,im3_eia_combined.csv,1147,50,1147,0,True,True
2,im3_multimodal_atlas_fueltype.csv,1147,21,1147,0,True,True
3,im3_multimodal_atlas_retail.csv,1147,12,1147,0,True,True


## unique columns check

In [41]:
for filename, df in datasets.items():
    print(f"\n{filename}")

    print("Columns:")
    for col in df.columns:
        print(f"  - {col}")


im3_baseline.csv
Columns:
  - id
  - state
  - state_abb
  - state_id
  - county
  - county_id
  - operator
  - ref
  - name
  - sqft
  - lon
  - lat
  - type

im3_eia_combined.csv
Columns:
  - id
  - state
  - state_abb
  - state_id
  - county
  - county_id
  - operator
  - ref
  - name
  - sqft
  - lon
  - lat
  - type
  - nearest_plant_km
  - plants_50km
  - capacity_50km_mw
  - renewable_50km_mw
  - renewable_share_50km
  - nearest_big_plant_km
  - ba_code
  - ba_peak_mw
  - ba_avg_mw
  - ba_peak_to_avg
  - ba_demand_coverage
  - ba_share_coal
  - ba_share_gas
  - ba_share_nuclear
  - ba_share_hydro
  - ba_share_solar
  - ba_share_wind
  - ba_share_fossil
  - ba_share_renewable
  - ba_share_clean
  - ba_total_gen_mwh
  - comm_elec_price_cents_kwh
  - comm_elec_sales_gwh
  - st_share_coal
  - st_share_gas
  - st_share_nuclear
  - st_share_hydro
  - st_share_solar
  - st_share_wind
  - st_share_fossil
  - st_share_renewable
  - st_share_clean
  - st_total_gen_mwh
  - est_mwh_per_hou

## prepping wri for spatial join

In [42]:
import geopandas as gpd
import pandas as pd

im3_eia = datasets["im3_eia_combined.csv"].copy()

# Preserve facility IDs
im3_eia["id"] = im3_eia["id"].astype("string")

# Check coordinates before creating geographic points
print("Missing lat:", im3_eia["lat"].isna().sum())
print("Missing lon:", im3_eia["lon"].isna().sum())
print("Duplicate IDs:", im3_eia["id"].duplicated().sum())

# Convert coordinates to geographic points
facilities = gpd.GeoDataFrame(
    im3_eia,
    geometry=gpd.points_from_xy(
        im3_eia["lon"],
        im3_eia["lat"]
    ),
    crs="EPSG:4326"
)

print("Total facilities:", len(facilities))
print("CRS:", facilities.crs)

display(facilities[["id", "lat", "lon", "geometry"]].head())

Missing lat: 0
Missing lon: 0
Duplicate IDs: 0
Total facilities: 1147
CRS: EPSG:4326


,id,lat,lon,geometry
0,0002744301,40.544256,-74.496521,POINT (-74.49652 40.54426)
1,0007805491,40.100657,-82.814358,POINT (-82.81436 40.10066)
2,0009474864,35.894738,-81.546515,POINT (-81.54651 35.89474)
3,0013924557,41.515955,-93.711719,POINT (-93.71172 41.51595)
4,0014593270,35.588771,-81.261809,POINT (-81.26181 35.58877)


## join

### do not need to rerun, only for initial

In [43]:
import geopandas as gpd
from pathlib import Path

wri_path = Path.cwd() / "Data" / "processed" / "wri_us_cleaned.parquet"

assert wri_path.is_file(), f"WRI Parquet not found: {wri_path}"

wri_us_geo = gpd.read_parquet(wri_path)

print("Loaded from:", wri_path.resolve())
print("Shape:", wri_us_geo.shape)
print("CRS:", wri_us_geo.crs.to_epsg())

score_cols_wri = [
    c for c in wri_us_geo.columns
    if c.endswith("_score")
]

print("Score columns:", len(score_cols_wri))
print("UCW present:", "ucw_score" in score_cols_wri)
print("RRI present:", "rri_score" in score_cols_wri)

Loaded from: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed/wri_us_cleaned.parquet
Shape: (3433, 24)
CRS: 4326
Score columns: 17
UCW present: True
RRI present: True


# check

In [44]:
print("Facility shape:", facilities.shape)
print("Unique IDs:", facilities["id"].nunique())
print("Duplicate IDs:", facilities["id"].duplicated().sum())

print("Missing coordinates:",
      facilities[["lat", "lon"]].isna().any(axis=1).sum())

print("Coordinates outside valid ranges:",
      (
          ~facilities["lat"].between(-90, 90) |
          ~facilities["lon"].between(-180, 180)
      ).sum())

print("CRS:", facilities.crs.to_epsg())
print("Existing WRI score columns:",
      [c for c in score_cols_wri if c in facilities.columns])

Facility shape: (1147, 51)
Unique IDs: 1147
Duplicate IDs: 0
Missing coordinates: 0
Coordinates outside valid ranges: 0
CRS: 4326
Existing WRI score columns: []


In [45]:
# Align WRI polygons with facility CRS
wri_us_geo = wri_us_geo.to_crs(facilities.crs)

# Spatial join: each facility receives its polygon's WRI attributes
wri_joined = gpd.sjoin(
    facilities,
    wri_us_geo,
    how="left",
    predicate="within"
)

print("Original facilities:", len(facilities))
print("Joined rows:", len(wri_joined))

print(
    "Unique matched facilities:",
    wri_joined.loc[
        wri_joined["index_right"].notna(), "id"
    ].nunique()
)

print(
    "Unmatched facilities:",
    wri_joined.loc[
        wri_joined["index_right"].isna(), "id"
    ].nunique()
)

print(
    "Facilities with multiple matches:",
    (wri_joined.groupby("id").size() > 1).sum()
)

Original facilities: 1147
Joined rows: 1147
Unique matched facilities: 1147
Unmatched facilities: 0
Facilities with multiple matches: 0


# check

In [46]:
joined_scores = [
    c for c in wri_joined.columns
    if c.endswith("_score")
]

print("Joined WRI scores:", len(joined_scores))
print("Missing WRI scores:",
      sorted(set(score_cols_wri) - set(joined_scores)))

print("Unexpected score columns:",
      sorted(set(joined_scores) - set(score_cols_wri)))

print("UCW present:", "ucw_score" in joined_scores)
print("RRI present:", "rri_score" in joined_scores)

print("Null polygon assignments:",
      wri_joined["index_right"].isna().sum())

Joined WRI scores: 17
Missing WRI scores: []
Unexpected score columns: []
UCW present: True
RRI present: True
Null polygon assignments: 0


## inspecting the matched risk values

In [47]:
score_cols = [
    col for col in wri_joined.columns
    if col.endswith("_score")
]

assert len(score_cols) == 17, (
    f"Expected 17 WRI indicators, found {len(score_cols)}"
)

display(
    wri_joined[
        ["id", "lat", "lon", "pfaf_id"] + score_cols[:5]
    ].head(10)
)

,id,lat,lon,pfaf_id,bws_score,bwd_score,iav_score,sev_score,gtd_score
0,0002744301,40.544256,-74.496521,731504,3.874239,1.728125,1.593892,0.905566,NaN
1,0007805491,40.100657,-82.814358,742678,2.834326,1.170674,1.012947,0.623887,NaN
2,0009474864,35.894738,-81.546515,732409,2.942697,1.164728,1.513840,0.295139,NaN
3,0013924557,41.515955,-93.711719,742843,0.612164,0.322639,3.525262,1.184212,NaN
4,0014593270,35.588771,-81.261809,732408,3.638327,1.419415,2.502750,0.332043,NaN
5,0014930068,35.049942,-106.542822,752195,3.577024,1.448563,1.658749,1.472403,NaN
6,0014997588,37.482763,-77.236455,731905,3.802848,1.485116,1.365430,0.316325,1.301487
7,0015884451,40.643753,-74.285481,731503,4.010867,1.566559,0.744697,0.144982,1.301487
8,0016282459,39.140242,-76.606431,731700,1.126948,0.512190,1.964271,1.302858,1.301487
9,0018518989,34.844123,-86.628423,742623,0.312350,0.285413,1.731731,1.014823,NaN


## check for missing spatial values

In [48]:
import pandas as pd

for col in score_cols:
    values = pd.to_numeric(
        wri_joined[col],
        errors="coerce"
    )

    sentinel_count = values.isin([-9999, 9999]).sum()

    if sentinel_count > 0:
        print(col, ":", sentinel_count, "sentinel values")

## validating the spatial merge

In [49]:
import pandas as pd

# 1. Spatial matching coverage
matched = wri_joined["index_right"].notna()

print("Total IM3 facilities:", facilities["id"].nunique())
print("Matched facilities:", wri_joined.loc[matched, "id"].nunique())
print("Unmatched facilities:", wri_joined.loc[~matched, "id"].nunique())

# 2. Duplicate spatial matches
duplicate_counts = wri_joined.groupby("id").size()
print("Facilities with multiple WRI polygons:",
      (duplicate_counts > 1).sum())

# 3. Missingness across WRI indicators
score_cols = [
    c for c in wri_joined.columns
    if c.endswith("_score")
]

wri_joined[score_cols] = (
    wri_joined[score_cols]
    .apply(pd.to_numeric, errors="coerce")
    .replace([-9999, 9999], pd.NA)
)

missingness = (
    wri_joined[score_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(missingness.to_frame("missing_pct"))

Total IM3 facilities: 1147
Matched facilities: 1147
Unmatched facilities: 0
Facilities with multiple WRI polygons: 0


,missing_pct
gtd_score,75.32694
bws_score,0.00000
bwd_score,0.00000
iav_score,0.00000
sev_score,0.00000
rfr_score,0.00000
cfr_score,0.00000
drr_score,0.00000
ucw_score,0.00000
cep_score,0.00000


## calling the parquet made in the final eda notebook

## final combined dataset

In [50]:
# Select WRI attributes for integration
wri_features = wri_joined[
    ["id", "pfaf_id"] + score_cols
].copy()

# Confirm one WRI record per facility
assert wri_features["id"].is_unique

# Merge WRI features into the existing IM3 + EIA dataset
combined = im3_eia.merge(
    wri_features,
    on="id",
    how="left",
    validate="one_to_one"
)

# Validate the combined dataset
assert len(combined) == len(im3_eia)
assert combined["id"].is_unique
assert combined["id"].notna().all()

print("Final shape:", combined.shape)
print("Unique IDs:", combined["id"].nunique())
print("WRI indicators:", len(score_cols))

display(combined.head(3))

Final shape: (1147, 68)
Unique IDs: 1147
WRI indicators: 17


,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,...,drr_score,ucw_score,cep_score,udw_score,usa_score,rri_score,w_awr_elp_qan_score,w_awr_elp_qal_score,w_awr_elp_rrr_score,w_awr_elp_tot_score
0,0002744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,...,1.720637,0.880683,4.037691,0.186522,0.0,0.88,2.945868,3.534384,1.050924,2.045674
1,0007805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,...,2.509289,0.880683,2.934937,0.273142,0.0,0.88,2.526726,2.450120,1.077413,1.633669
2,0009474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,...,1.975714,0.880683,1.847858,1.254066,0.0,0.88,2.520564,1.272564,1.377390,1.567883


In [51]:
from pathlib import Path

output_path = Path("Data/processed/im3_eia_wri_combined_17scores.csv")

assert combined.shape == (1147, 68)
assert combined["id"].is_unique
assert len(score_cols) == 17

output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists():
    print("File already exists. Export skipped:", output_path.resolve())
else:
    combined.to_csv(output_path, index=False)
    print("Saved:", output_path.resolve())

File already exists. Export skipped: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed/im3_eia_wri_combined_17scores.csv


In [52]:
import pandas as pd

saved_combined = pd.read_csv(
    output_path,
    dtype={"id": str}
)

expected = combined.sort_values("id").reset_index(drop=True)
actual = saved_combined.sort_values("id").reset_index(drop=True)

pd.testing.assert_frame_equal(
    expected,
    actual[expected.columns],
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("Local merged CSV matches validated dataset: PASS")
print("Shape:", actual.shape)
print("Unique IDs:", actual["id"].nunique())
print("WRI indicators:", len([
    c for c in actual.columns if c.endswith("_score")
]))

Local merged CSV matches validated dataset: PASS
Shape: (1147, 68)
Unique IDs: 1147
WRI indicators: 17


## upload to s3

### only needed for initial run

import boto3

s3 = boto3.client("s3")

bucket = "atlas-acm-research-f26"
key = "dataset/im3_eia_wri_combined.csv"

s3.upload_file(
    str(output_path),
    bucket,
    key
)

print(f"Uploaded to s3://{bucket}/{key}")

## final check to see if original facility data is preserved 

In [53]:
import pandas as pd

original = im3_eia.set_index("id").sort_index()
merged_original = combined.set_index("id").sort_index()

assert set(original.index) == set(merged_original.index)
assert all(c in combined.columns for c in im3_eia.columns)

pd.testing.assert_frame_equal(
    original,
    merged_original[original.columns],
    check_dtype=False
)

assert all(c in combined.columns for c in score_cols)
assert combined["id"].is_unique

print("Original IM3/EIA attributes preserved: PASS")
print("All 17 WRI scores present: PASS")
print("Final shape:", combined.shape)

Original IM3/EIA attributes preserved: PASS
All 17 WRI scores present: PASS
Final shape: (1147, 68)


### this is only for initial run, dont need to rerun this

In [54]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")

bucket = "atlas-acm-research-f26"
key = "dataset/im3_eia_wri_combined.csv"

response = s3.get_object(Bucket=bucket, Key=key)

s3_df = pd.read_csv(
    BytesIO(response["Body"].read()),
    dtype={"id": str}
)

s3_scores = [c for c in s3_df.columns if c.endswith("_score")]

print("S3 shape:", s3_df.shape)
print("S3 score columns:", len(s3_scores))
print("UCW present:", "ucw_score" in s3_scores)
print("RRI present:", "rri_score" in s3_scores)
print("Unique IDs:", s3_df["id"].nunique())
print("Last modified:", response["LastModified"])

S3 shape: (1147, 68)
S3 score columns: 17
UCW present: True
RRI present: True
Unique IDs: 1147
Last modified: 2026-10-19 21:36:02+00:00


In [55]:
expected = combined.sort_values("id").reset_index(drop=True)
actual = s3_df.sort_values("id").reset_index(drop=True)

pd.testing.assert_frame_equal(
    expected,
    actual[expected.columns],
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("S3 matches validated combined dataset: PASS")
print("Rows:", len(actual))
print("Columns:", len(actual.columns))

S3 matches validated combined dataset: PASS
Rows: 1147
Columns: 68


## wri only facility feature table

**Feature groups:** helps with looking at what group could help with identification the best

In [56]:
wri_feature_cols = ["id", "pfaf_id"] + score_cols

print("Missing required columns:",
      sorted(set(wri_feature_cols) - set(combined.columns)))

wri_features = combined[wri_feature_cols].copy()

print("Feature table shape:", wri_features.shape)
print("Unique facility IDs:", wri_features["id"].nunique())
print("Duplicate facility IDs:", wri_features["id"].duplicated().sum())
print("Missing watershed IDs:", wri_features["pfaf_id"].isna().sum())

Missing required columns: []
Feature table shape: (1147, 19)
Unique facility IDs: 1147
Duplicate facility IDs: 0
Missing watershed IDs: 0


In [57]:
from pathlib import Path

feature_path = Path(
    "Data/processed/wri_aqueduct_facility_features_17scores.csv"
)

assert wri_features.shape == (1147, 19)
assert wri_features["id"].is_unique
assert wri_features["pfaf_id"].notna().all()

feature_path.parent.mkdir(parents=True, exist_ok=True)

if feature_path.exists():
    print("File already exists. Export skipped:", feature_path.resolve())
else:
    wri_features.to_csv(feature_path, index=False)
    print("Saved:", feature_path.resolve())

print("Feature table shape:", wri_features.shape)

File already exists. Export skipped: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed/wri_aqueduct_facility_features_17scores.csv
Feature table shape: (1147, 19)


## validated saved csv

In [58]:
import pandas as pd

saved_features = pd.read_csv(
    feature_path,
    dtype={"id": str, "pfaf_id": str}
)

expected_features = wri_features.copy()
expected_features["id"] = expected_features["id"].astype(str)
expected_features["pfaf_id"] = expected_features["pfaf_id"].astype(str)

pd.testing.assert_frame_equal(
    expected_features.reset_index(drop=True),
    saved_features[expected_features.columns].reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("Saved CSV matches in-memory WRI features: PASS")
print("Shape:", saved_features.shape)
print("Unique facility IDs:", saved_features["id"].nunique())
print("WRI indicators:", len([
    c for c in saved_features.columns if c.endswith("_score")
]))

Saved CSV matches in-memory WRI features: PASS
Shape: (1147, 19)
Unique facility IDs: 1147
WRI indicators: 17


## WRI-only S3 export
**1:**The validated 17-score WRI feature table is saved locally at Data/processed/wri_aqueduct_facility_features_17scores.csv.

**2:**S3 upload is intentionally deferred. The full IM3/EIA + WRI dataset has already been uploaded and validated separately.

**3:**Any future WRI-only upload should use an explicitly approved S3 key and be verified after upload.

## upload

### only needed for initial run

In [59]:
import boto3
from botocore.exceptions import ClientError

s3 = boto3.client("s3")

bucket = "atlas-acm-research-f26"
key = "features/wri_aqueduct_facility_features_17scores.csv"

assert feature_path.is_file(), f"Local file not found: {feature_path}"

try:
    s3.head_object(Bucket=bucket, Key=key)
    print("S3 object already exists. Upload skipped.")

except ClientError as e:
    error_code = e.response["Error"]["Code"]

    if error_code in ("404", "NoSuchKey", "NotFound"):
        s3.upload_file(
            str(feature_path),
            bucket,
            key
        )
        print(f"Uploaded to s3://{bucket}/{key}")
    else:
        raise

S3 object already exists. Upload skipped.


In [60]:
import boto3

s3 = boto3.client("s3")

bucket = "atlas-acm-research-f26"
key = "features/wri_aqueduct_facility_features_17scores.csv"

# Prevent overwriting an existing object
s3.head_object(Bucket=bucket, Key=key)

{'ResponseMetadata': {'RequestId': 'GWBZTR24VQ82SQQZ',
  'HostId': '6+z31cxVxS7uvaGCwyJRzqd9NgZimijxDIkMa+mFwW7f8C4Fft1KlhK1Njr+2scsPVuGrJ/NnZXJHxmhK1kMSXJhIb74Twy3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '6+z31cxVxS7uvaGCwyJRzqd9NgZimijxDIkMa+mFwW7f8C4Fft1KlhK1Njr+2scsPVuGrJ/NnZXJHxmhK1kMSXJhIb74Twy3',
   'x-amz-request-id': 'GWBZTR24VQ82SQQZ',
   'server': 'AmazonS3',
   'date': 'Mon, 19 Oct 2026 22:26:48 GMT',
   'last-modified': 'Mon, 19 Oct 2026 21:58:30 GMT',
   'etag': '"4b9e141f46a5ff3a5000066128bfa754"',
   'x-amz-server-side-encryption': 'AES256',
   'accept-ranges': 'bytes',
   'content-length': '290248',
   'content-type': 'binary/octet-stream'},
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2026, 10, 19, 21, 58, 30, tzinfo=tzutc()),
 'ContentLength': 290248,
 'ETag': '"4b9e141f46a5ff3a5000066128bfa754"',
 'ContentType': 'binary/octet-stream',
 'ServerSideEncryption': 'AES256',
 'Metadata': {}}

# validate

In [61]:
from io import BytesIO
import pandas as pd

response = s3.get_object(
    Bucket="atlas-acm-research-f26",
    Key="features/wri_aqueduct_facility_features_17scores.csv"
)

s3_features = pd.read_csv(
    BytesIO(response["Body"].read()),
    dtype={"id": str, "pfaf_id": str}
)

expected = wri_features.copy()
expected["id"] = expected["id"].astype(str)
expected["pfaf_id"] = expected["pfaf_id"].astype(str)

pd.testing.assert_frame_equal(
    expected.reset_index(drop=True),
    s3_features[expected.columns].reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("S3 WRI features match validated dataset: PASS")
print("Shape:", s3_features.shape)
print("Unique facility IDs:", s3_features["id"].nunique())
print("WRI indicators:", len([
    c for c in s3_features.columns if c.endswith("_score")
]))

S3 WRI features match validated dataset: PASS
Shape: (1147, 19)
Unique facility IDs: 1147
WRI indicators: 17


## looking at facilities for spatial join

In [62]:
print("Shape:", facilities.shape)
print("CRS:", facilities.crs)
print("Unique IDs:", facilities["id"].nunique())

print("Existing WRI scores:",
      [c for c in facilities.columns
       if c.endswith("_score")])

Shape: (1147, 51)
CRS: EPSG:4326
Unique IDs: 1147
Existing WRI scores: []
